# 07-03 · ICR 분모 후보 3개 비교 (이자비용 / 금융비용 / 현금 이자지급)

| 항목 | 내용 |
|---|---|
| 하는 일 | 2019·2020 사업보고서에서 분모 후보 세 금액을 읽어 후보끼리의 비율 분포를 센다 — 측정 타당성만 확인 |
| 읽는 파일 | `outputs/04_본수집/본수집_로그.csv`, `본수집_결과_기업별.csv`, 저장된 응답 |
| 만드는 파일 | `outputs/05_계정확보/이자분모_후보_비교_연간.csv` |
| 인증키 | 필요 없음 (저장된 파일만 읽습니다) |
| 걸리는 시간 | 약 12초 — 2026-11-05, 이 컴퓨터에서 측정 |

> 이 노트북이 **원본**입니다. 같은 이름의 `.py`는 `scripts/노트북에서_py_만들기.py`가 여기서 자동으로 만든 파일이라 직접 고치지 않습니다. 원래 손으로 쓴 `.py`는 이 폴더의 `원본_스크립트/`에 보관했습니다.

영업이익·노출도 E는 읽지 않고 ICR도 계산하지 않아 '결과(ICR 변화)와 무관한 측정 점검'입니다. 한 응답에서 같은 후보가 여러 줄 잡히면 합치지 않고 '여러 줄'로만 세어 비율 계산에서 뺍니다. 이 비교는 ICR 분모를 무엇으로 할지(팀원 v1 정의와 맞출지 포함) **팀과 정할 때의 참고 자료**이며 분모는 아직 팀 확정이 아닙니다.

### 준비 — 불러오기와 경로

필요한 도구를 불러오고, 이 폴더 위치(`HERE`)를 잡습니다. 맨 위 따옴표 글(독스트링)은 이 스크립트의 원래 사용설명서입니다.

In [1]:
"""ICR 분모 후보(이자비용 / 금융비용 / 현금흐름표 이자지급)가 서로 얼마나 다른지 측정 타당성만 본다.

이 스크립트가 하는 일
  - 사전기간 연간 보고서(2019·2020 사업보고서, 상태 000)에서 세 후보 금액을 읽어, 후보끼리의 비율 분포를 센다.
  - 영업이익·노출도 E는 읽지 않는다. ICR을 계산하지 않는다. 즉 결과(ICR 변화)와 무관한 '측정' 점검이다.
  - 계정 이름 규칙은 config.ACCOUNT_PATTERNS 그대로(결과를 보고 고치지 않는다).
  - 한 응답에서 같은 후보가 여러 줄 매칭되는 경우는 합치지 않고 '여러 줄'로만 세고 비율 계산에서는 뺀다.
저장: outputs/05_계정확보/이자분모_후보_비교_연간.csv  (기업·연도별 세 후보 금액)
실행: python -X utf8 03_이자분모_후보_비교.py (저장된 파일만 읽음, 키 필요 없음)
"""
import csv
import json
import re
import statistics
import sys
from collections import Counter
from pathlib import Path
try:
    HERE = HERE      # .py로 실행할 때
except NameError:
    HERE = Path.cwd()                           # 노트북에서 실행할 때(노트북이 있는 폴더)
sys.path.insert(0, str(HERE.parent / "01_파일럿"))
import config  # noqa: E402
if hasattr(sys.stdout, "reconfigure"):          # 노트북의 출력 창에는 이 함수가 없다
    sys.stdout.reconfigure(encoding="utf-8")

### 상수 — 입력·출력 경로와 규칙 값

In [2]:
LOG = config.BASE / "outputs" / "04_본수집" / "본수집_로그.csv"
RESULT = config.BASE / "outputs" / "04_본수집" / "본수집_결과_기업별.csv"
OUTDIR = config.BASE / "outputs" / "05_계정확보"
FS_DIR = config.RAW / "fs"
KEYS = {"interest_exp": "이자비용", "fin_cost": "금융비용", "interest_paid_cf": "이자지급_CF"}
PATS = {k: (set(config.ACCOUNT_PATTERNS[k][0].split("|")), re.compile(config.ACCOUNT_PATTERNS[k][1])) for k in KEYS}

#### `to_num()`

In [3]:
def to_num(v):
    try:
        return float(str(v).replace(",", "").strip())
    except ValueError:
        return None

#### `pick()`

In [4]:
def pick(rows, key):
    sjs, pat = PATS[key]
    hits = [r for r in rows if r.get("sj_div") in sjs and pat.search(re.sub(r"\s+", "", r.get("account_nm", "")))
            and to_num(r.get("thstrm_amount")) is not None]
    return hits

#### `q()`

In [5]:
def q(vals, p):
    vals = sorted(vals)
    return vals[min(len(vals) - 1, int(round(p * (len(vals) - 1))))]

#### `describe()`

In [6]:
def describe(name, ratios):
    if not ratios:
        print(f"  {name}: 해당 없음")
        return
    print(f"  {name}: n={len(ratios):,} | 10%={q(ratios,.1):.2f} 25%={q(ratios,.25):.2f} 중앙값={statistics.median(ratios):.2f} 75%={q(ratios,.75):.2f} 90%={q(ratios,.9):.2f}")

#### `main()`

In [7]:
def main():
    firms = {r["corp_code"]: r for r in csv.DictReader(open(RESULT, encoding="utf-8-sig"))}
    chosen = {}
    for r in csv.DictReader(open(LOG, encoding="utf-8-sig")):
        if r["status"] == "000" and r["report"] == "FY" and int(r["year"]) in (2019, 2020):
            k = (r["corp_code"], int(r["year"]))
            if k not in chosen or r["fs_div"] == "CFS":
                chosen[k] = r["fs_div"]
    rows = []
    multi = Counter()
    for (c, y), fs in sorted(chosen.items()):
        d = json.loads((FS_DIR / f"{c}_{y}_11011_{fs}.json").read_text(encoding="utf-8"))["list"]
        rec = {"corp_code": c, "회사명": firms[c]["회사명"], "현재_법인구분": firms[c]["현재_법인구분"], "year": y, "fs_div": fs}
        for k, nm in KEYS.items():
            h = pick(d, k)
            if len(h) > 1:
                multi[k] += 1
                rec[nm] = "여러줄"
            elif len(h) == 1:
                rec[nm] = to_num(h[0]["thstrm_amount"])
            else:
                rec[nm] = ""
        rows.append(rec)
    OUTDIR.mkdir(parents=True, exist_ok=True)
    with open(OUTDIR / "이자분모_후보_비교_연간.csv", "w", encoding="utf-8-sig", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)

    def num(r, nm):
        v = r[nm]
        return v if isinstance(v, float) else None

    print(f"연간 보고서 {len(rows):,}건(2019·2020 사업보고서, 기업 {len({r['corp_code'] for r in rows}):,}곳)")
    print("후보가 한 응답에서 여러 줄 매칭된 건수:", dict(multi))
    have = {nm: sum(1 for r in rows if num(r, nm) is not None) for nm in KEYS.values()}
    print("후보별로 값이 하나로 잡힌 건수:", have)

    print("\n[A] 이자비용과 금융비용이 둘 다 잡힌 건: 이자비용 ÷ 금융비용 (1이면 같은 값, 작을수록 금융비용에 이자 아닌 항목이 많음)")
    both = [(num(r, "이자비용"), num(r, "금융비용")) for r in rows if num(r, "이자비용") and num(r, "금융비용") and num(r, "금융비용") > 0]
    describe("이자비용/금융비용", [a / b for a, b in both])
    print("[B] 이자비용과 CF 이자지급이 둘 다 잡힌 건: CF 이자지급 ÷ 이자비용")
    both2 = [(num(r, "이자지급_CF"), num(r, "이자비용")) for r in rows if num(r, "이자지급_CF") and num(r, "이자비용") and num(r, "이자비용") > 0]
    describe("이자지급/이자비용", [a / b for a, b in both2])
    print("[C] 금융비용과 CF 이자지급이 둘 다 잡힌 건(전체 표본): CF 이자지급 ÷ 금융비용")
    both3 = [(num(r, "이자지급_CF"), num(r, "금융비용")) for r in rows if num(r, "이자지급_CF") and num(r, "금융비용") and num(r, "금융비용") > 0]
    ratios = [a / b for a, b in both3]
    describe("이자지급/금융비용", ratios)
    n = len(ratios)
    for lo, hi in ((0, .25), (.25, .5), (.5, .75), (.75, 1.25), (1.25, 99)):
        k = sum(1 for x in ratios if lo <= x < hi)
        print(f"     {lo:.2f}~{hi:.2f}: {k:,}건 ({k/n:.1%})")
    print("\n이자비용이 값으로 잡힌 기업:", len({r['corp_code'] for r in rows if num(r, '이자비용') is not None}), "곳")
    print("저장:", OUTDIR / "이자분모_후보_비교_연간.csv")

### (.py에만 들어가는 부분)

노트북에서는 `__name__`이 항상 `"__main__"`이라 아래 코드를 일반 셀로 두면 열 때마다 실행됩니다. 그래서 실행하지 않는 설명 셀에 두고, 변환 스크립트가 `.py`에만 넣습니다.

```python
if __name__ == "__main__":
    main()
```

In [8]:
main()

연간 보고서 2,642건(2019·2020 사업보고서, 기업 1,321곳)
후보가 한 응답에서 여러 줄 매칭된 건수: {'interest_paid_cf': 23, 'interest_exp': 2, 'fin_cost': 2}
후보별로 값이 하나로 잡힌 건수: {'이자비용': 118, '금융비용': 2619, '이자지급_CF': 2501}

[A] 이자비용과 금융비용이 둘 다 잡힌 건: 이자비용 ÷ 금융비용 (1이면 같은 값, 작을수록 금융비용에 이자 아닌 항목이 많음)
  이자비용/금융비용: n=114 | 10%=0.03 25%=0.18 중앙값=0.56 75%=0.98 90%=1.00
[B] 이자비용과 CF 이자지급이 둘 다 잡힌 건: CF 이자지급 ÷ 이자비용
  이자지급/이자비용: n=106 | 10%=0.13 25%=0.54 중앙값=0.94 75%=1.00 90%=1.03
[C] 금융비용과 CF 이자지급이 둘 다 잡힌 건(전체 표본): CF 이자지급 ÷ 금융비용
  이자지급/금융비용: n=2,420 | 10%=0.03 25%=0.13 중앙값=0.42 75%=0.80 90%=1.00
     0.00~0.25: 806건 (33.3%)
     0.25~0.50: 478건 (19.8%)
     0.50~0.75: 377건 (15.6%)
     0.75~1.25: 639건 (26.4%)
     1.25~99.00: 50건 (2.1%)

이자비용이 값으로 잡힌 기업: 66 곳
저장: C:\Users\SAMSUNG\OneDrive\바탕 화면\공모전_26-2 BAF\BAF-26-2-finance_2\BAF-26-2-finance_2\Corp\outputs\05_계정확보\이자분모_후보_비교_연간.csv


### 저장된 결과 미리보기 (맨 위 5행)

위 실행이 만들었거나 이미 저장돼 있던 결과 파일을 표로 엽니다. 파일 전체는 엑셀로 여세요.

In [9]:
import pandas as pd
from IPython.display import display

_p = HERE.parents[1] / "outputs/05_계정확보/이자분모_후보_비교_연간.csv"
print(_p.name, "|", end=" ")
_df = pd.read_csv(_p, encoding="utf-8-sig", low_memory=False)
print(f"{len(_df):,}행 × {_df.shape[1]}열")
display(_df.head(5))

이자분모_후보_비교_연간.csv | 2,642행 × 8열


,corp_code,회사명,현재_법인구분,year,fs_div,이자비용,금융비용,이자지급_CF
0,100601,강원에너지,K,2019,CFS,NaN,7834726712.0,451752394.0
1,100601,강원에너지,K,2020,CFS,NaN,7028652433.0,468940494.0
2,100939,강남제비스코,Y,2019,CFS,NaN,554425372.0,191890127.0
3,100939,강남제비스코,Y,2020,CFS,NaN,1052851374.0,250668555.0
4,101044,에이프로젠바이오로직스,Y,2019,CFS,NaN,4236029276.0,700167281.0
